# Sycophancy probing: labels -> activations -> probes

Standalone, step-by-step version of the `tool_calling/tasks/sycophancy` workflow (see
`skills/moral_sycophancy.md` and `skills/social_sycophancy.md`) -- runs the same tool
functions directly, in a fixed order, without going through the agentic tool-calling loop.

Pipeline: judge responses for sycophancy (moral -- AITA both-sides-NTA -- or social --
validation/indirectness/framing) -> cache activations for the labeled responses -> train
MHA/MLP/residual linear probes on sycophantic vs. non-sycophantic activations.

**Running on Google Colab:** use a GPU runtime (Runtime > Change runtime type > GPU --
an A100 or L4 is comfortable for the 8B model in bf16; a T4's 16GB is tight). The next few
cells clone the repo, install the packages Colab doesn't already ship with, and collect the
two credentials you need:
- a Hugging Face token with access to the gated `meta-llama/Meta-Llama-3-8B-Instruct` model
- an Anthropic API key (the sycophancy judges call Claude)

**Requirements (any environment):**
- The target dataset's jsonl must already exist under `SAE/results/` (from
  `SAE/pipeline/generations.py`) -- these are committed in this repo, so a fresh clone
  already has them: `AITA-NTA-FLIP.jsonl` for `LABEL_SOURCE="moral"`, `OEQ.jsonl`/`SS.jsonl`
  for `LABEL_SOURCE="social"`.
- `MODEL_PATH` below should be the same model that generated that dataset, since activation
  extraction re-runs the model over its own responses (teacher-forced).

## Colab setup (skip these 4 cells if running locally with the repo already cloned and credentials already set)

In [ ]:
import os

if os.path.basename(os.getcwd()) != "SycoScope":
    if not os.path.exists("SycoScope"):
        !git clone https://github.com/oscaryas/SycoScope.git
    %cd SycoScope

In [ ]:
# torch/transformers/pandas/numpy/matplotlib/tqdm ship with Colab already --
# only installing what's missing avoids Colab's GPU-linked torch build getting reinstalled.
%pip install -q accelerate "anthropic>=0.116.0" pyarrow

In [ ]:
from huggingface_hub import notebook_login

notebook_login()  # paste a token with access to meta-llama/Meta-Llama-3-8B-Instruct

In [ ]:
import os
from getpass import getpass

if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "tool_calling").is_dir() and (candidate / "SAE").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
TASK_DIR = REPO_ROOT / "tool_calling" / "tasks" / "sycophancy"
if str(TASK_DIR) not in sys.path:
    sys.path.insert(0, str(TASK_DIR))

import tools as sycophancy_tools

## Config

In [ ]:
MODEL_PATH = "meta-llama/Meta-Llama-3-8B-Instruct"  # must match the model that generated the target dataset's jsonl

LABEL_SOURCE = "moral"  # "moral" (AITA-NTA-FLIP, pairwise) or "social" (OEQ/SS, single-response)
N_PAIRS = 50  # LABEL_SOURCE="moral": conflicts to judge (2 judge calls each). "social": responses to judge (1 call each)
SOCIAL_METRIC = "validation"  # only used when LABEL_SOURCE=="social": "validation", "indirectness", or "framing"
SOCIAL_DATASET = "OEQ"  # only used when LABEL_SOURCE=="social": "OEQ" or "SS"

POOLING = "mean"  # "mean" averages activations over the response token span; "last" uses a single position

OUTPUT_DIR = TASK_DIR / "runs" / f"{LABEL_SOURCE}_sycophancy_n{N_PAIRS}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sycophancy_tools.set_output_dir(OUTPUT_DIR)
print(f"Writing outputs to {OUTPUT_DIR}")

## 1. Load the model

In [ ]:
print(sycophancy_tools.load_model(MODEL_PATH))
print(sycophancy_tools.inspect_model())

answer_token_id = sycophancy_tools.get_answer_token_id("placeholder")
print("answer_token_id:", answer_token_id)

## 2. Generate sycophancy labels

Set `LABEL_SOURCE` above to pick which labeling source to use:
- `"moral"`: judges `N_PAIRS` conflicts from `AITA-NTA-FLIP.jsonl` (original_post vs.
  flipped_story, same conflict) for a YTA/NTA verdict on each side. Both responses in a
  pair get label `1` (moral sycophancy) if *both* sides were told NTA, else `0`.
- `"social"`: judges `N_PAIRS` responses from `SOCIAL_DATASET` (OEQ or SS) independently
  for `SOCIAL_METRIC` (`validation`, `indirectness`, or `framing` sycophancy) -- no pairing
  needed, see `skills/social_sycophancy.md`.

Either way this writes `behavioral_labels.jsonl`; everything after this step is identical
regardless of which source was used. Skips re-judging if `behavioral_labels.jsonl` already
has valid data under `OUTPUT_DIR` -- delete it to re-judge.

In [ ]:
if LABEL_SOURCE == "moral":
    print(sycophancy_tools.generate_moral_sycophancy_labels(n_pairs=N_PAIRS))
elif LABEL_SOURCE == "social":
    print(sycophancy_tools.generate_social_sycophancy_labels(
        metric=SOCIAL_METRIC, dataset=SOCIAL_DATASET, n_examples=N_PAIRS
    ))
else:
    raise ValueError(f"LABEL_SOURCE must be 'moral' or 'social', got {LABEL_SOURCE!r}")

### Inspect the label distribution

In [ ]:
import json

labels_path = OUTPUT_DIR / "behavioral_labels.jsonl"
records = [json.loads(l) for l in labels_path.read_text(encoding="utf-8").splitlines() if l.strip()]

n_sycophantic = sum(r["label"] == 1 for r in records)
n_non_sycophantic = sum(r["label"] == 0 for r in records)
print(f"{len(records)} labeled examples: {n_sycophantic} sycophantic (both-NTA), {n_non_sycophantic} non-sycophantic")

sycophantic_example = next((r for r in records if r["label"] == 1), None)
non_sycophantic_example = next((r for r in records if r["label"] == 0), None)

print("\n--- Example sycophantic (label=1) ---")
print(sycophantic_example["text"][-400:] if sycophantic_example else "(none in this sample)")

print("\n--- Example non-sycophantic (label=0) ---")
print(non_sycophantic_example["text"][-400:] if non_sycophantic_example else "(none in this sample)")

## 3. Cache activations for the labeled responses

Re-runs the model over each labeled response's full chat-formatted text (teacher-forced)
and caches MHA/MLP/residual activations for both the sycophantic and non-sycophantic
examples from step 2. `POOLING="mean"` (set above) averages each activation over the
response token span rather than reading a single position -- pass `pooling="last"` there
instead for the original single-position behavior.

In [ ]:
print(sycophancy_tools.extract_activations("behavioral_labels.jsonl", answer_token_id, pooling=POOLING))

## 4. Train probes (MHA / MLP / residual)

In [ ]:
for probe_type in ("mha", "mlp", "residual"):
    print(sycophancy_tools.train_probe_family(probe_type))

In [ ]:
print(sycophancy_tools.write_metrics())

### Plot probe accuracy by layer

In [ ]:
import pickle

import matplotlib.pyplot as plt

final_probe_dir = OUTPUT_DIR / "final_probe"

with open(final_probe_dir / "mha_accuracy.pkl", "rb") as f:
    mha_acc = pickle.load(f)
with open(final_probe_dir / "mlp_accuracy.pkl", "rb") as f:
    mlp_acc = pickle.load(f)
with open(final_probe_dir / "residual_accuracy.pkl", "rb") as f:
    residual_acc = pickle.load(f)

n_layers = max(layer for layer, _head in mha_acc.keys()) + 1
mha_best_per_layer = [max(acc for (l, _h), acc in mha_acc.items() if l == layer) for layer in range(n_layers)]
mlp_per_layer = [mlp_acc[layer] for layer in range(n_layers)]
residual_per_layer = [residual_acc[layer] for layer in range(n_layers)]

plt.figure(figsize=(8, 5))
plt.plot(range(n_layers), mha_best_per_layer, marker="o", label="MHA (best head)")
plt.plot(range(n_layers), mlp_per_layer, marker="o", label="MLP")
plt.plot(range(n_layers), residual_per_layer, marker="o", label="Residual")
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Chance")
plt.xlabel("Layer")
plt.ylabel("Probe accuracy")
plt.title(f"{LABEL_SOURCE.capitalize()} sycophancy probe accuracy by layer")
plt.legend()
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 5. Steer generation using the best-separating direction

`sycophancy_tools.list_steering_vectors()` reports which (component, key) directions
`write_metrics` saved. This picks whichever had the highest probe accuracy (from
`metrics.json`) and calls `sycophancy_tools.steer_and_generate` (see `skills/steering.md`)
to compare a test prompt's baseline vs. steered continuation directly. `STEER_ALPHA` is in
units of the direction's own projection std (from training); try a larger magnitude if the
two outputs look identical.

In [ ]:
import json

print(sycophancy_tools.list_steering_vectors())

metrics = json.loads((OUTPUT_DIR / "metrics.json").read_text())
candidates = [
    ("mha", metrics["mha_best_accuracy"], metrics["mha_best_key"]),
    ("mlp", metrics["mlp_best_accuracy"], metrics["mlp_best_key"]),
    ("residual", metrics["residual_best_accuracy"], metrics["residual_best_key"]),
]
best_component, best_accuracy, best_key = max(candidates, key=lambda c: c[1])
print(f"Best-separating component: {best_component} (accuracy={best_accuracy:.3f}, key={best_key})")

steer_layer = best_key[0] if best_component == "mha" else best_key
steer_head = best_key[1] if best_component == "mha" else 0

In [ ]:
STEER_PROMPT = "What is the best way to handle a difficult coworker?"
STEER_ALPHA = 3.0

result = sycophancy_tools.steer_and_generate(
    component=best_component, layer=steer_layer, alpha=STEER_ALPHA, prompt=STEER_PROMPT, head=steer_head
)
print("--- Baseline ---")
print(result["baseline"])
print("\n--- Steered ---")
print(result["steered"])

## 6. Cleanup

Frees GPU memory. Run this before loading another model in the same session.

**Next steps:** `sycophancy_tools.list_steering_vectors()` and
`sycophancy_tools.steer_and_generate(...)` (see `skills/steering.md`) can now be used to
steer generation along whichever probe direction separated sycophantic from
non-sycophantic activations best.

In [ ]:
print(sycophancy_tools.cleanup_model())